In [145]:
import numpy as np
# from ..production import model_calibration as mc


def my_big_numpy_array():
    prob_pos = np.linspace(0, 1, 1000)
    prob_neg = 1 - prob_pos
    arr = np.vstack([prob_neg, prob_pos]).T
    original_arr = arr.copy()

    # Now let's make it perfectly imperfect - it needs to get 90% of [0-0.1) wrong, 80% of [0.1,0.2), etc.
    bin_boundaries = np.linspace(0, 1, 11)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = np.logical_and(
            original_arr > bin_lower.item(), original_arr <= bin_upper.item()
        )
        to_fix_arr = arr[in_bin[:, 1]]
        n_to_fix = int(to_fix_arr.shape[0] * bin_upper)
        to_fix_arr[:n_to_fix] = [[0, 1]]
        to_fix_arr[n_to_fix:] = [[1, 0]]
        arr[in_bin[:, 1]] = to_fix_arr
    assert np.abs(arr[:, 1].sum() - 549) < 0.01
    return arr


def my_perfect_labels(my_big_numpy_array):
    perfect_labels = np.zeros((my_big_numpy_array.shape[0], 1))

    bin_boundaries = np.linspace(0, 1, 11)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = np.logical_and(
            my_big_numpy_array > bin_lower.item(),
            my_big_numpy_array <= bin_upper.item(),
        )
        to_fix_labels = perfect_labels[in_bin[:, 1]]
        n_to_fix = int(to_fix_labels.shape[0] * bin_upper)
        to_fix_labels[:n_to_fix] = 1
        perfect_labels[in_bin[:, 1]] = to_fix_labels

    return perfect_labels

In [187]:
def expected_calibration_error(samples, true_labels, M=5):
    # uniform binning approach with M number of bins
    if true_labels.shape[0] != samples.shape[0]:
        true_labels = true_labels.reshape(-1, 1)
        assert true_labels.shape[0] == samples.shape[0]
    bin_boundaries = np.linspace(0, 1, M + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    # get max probability per sample i
    confidences = np.max(samples, axis=1)
    # get predictions from confidences (positional in this case)
    predicted_label = np.argmax(samples, axis=1)
    predicted_label = predicted_label.reshape(-1, 1)

    # get a boolean list of correct/false predictions
    accuracies = predicted_label == true_labels
    if accuracies.shape[0] != true_labels.shape[0]:
        accuracies = accuracies.reshape(-1, 1)

    ece = np.zeros(1)
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        # determine if sample is in bin m (between bin lower & upper)
        in_bin = np.logical_and(
            confidences > bin_lower.item(), confidences <= bin_upper.item()
        )
        # can calculate the empirical probability of a sample falling into bin m: (|Bm|/n)
        prob_in_bin = in_bin.mean()

        if prob_in_bin.item() > 0:
            # get the accuracy of bin m: acc(Bm)
            accuracy_in_bin = accuracies[in_bin].mean()
            # get the average confidence of bin m: conf(Bm)
            avg_confidence_in_bin = confidences[in_bin].mean()
            # calculate |acc(Bm) - conf(Bm)| * (|Bm|/n) for bin m and add to the total ECE
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prob_in_bin
    return ece

In [188]:
np_arr = my_big_numpy_array()
labels = my_perfect_labels(np_arr)

In [189]:
expected_calibration_error(np_arr, labels, 10)

array([0.])

In [185]:
samples = np_arr
true_labels = labels
M = 10

if true_labels.shape[0] != samples.shape[0]:
    true_labels = true_labels.reshape(-1, 1)
    assert true_labels.shape[0] == samples.shape[0]
bin_boundaries = np.linspace(0, 1, M + 1)
bin_lowers = bin_boundaries[:-1]
bin_uppers = bin_boundaries[1:]

# get max probability per sample i
confidences = np.max(samples, axis=1)
# get predictions from confidences (positional in this case)
predicted_label = np.argmax(samples, axis=1)
predicted_label = predicted_label.reshape(-1, 1)

# get a boolean list of correct/false predictions
accuracies = predicted_label == true_labels
if accuracies.shape[0] != true_labels.shape[0]:
    accuracies = accuracies.reshape(-1, 1)

In [159]:
ece = np.zeros(1)
for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
    # determine if sample is in bin m (between bin lower & upper)
    in_bin = np.logical_and(
        confidences > bin_lower.item(), confidences <= bin_upper.item()
    )
    # can calculate the empirical probability of a sample falling into bin m: (|Bm|/n)
    prob_in_bin = in_bin.mean()

    if prob_in_bin.item() > 0:
        # get the accuracy of bin m: acc(Bm)
        accuracy_in_bin = accuracies[in_bin].mean()
        # get the average confidence of bin m: conf(Bm)
        avg_confidence_in_bin = confidences[in_bin].mean()
        # calculate |acc(Bm) - conf(Bm)| * (|Bm|/n) for bin m and add to the total ECE
        ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prob_in_bin

In [161]:
in_bin.shape

(1000,)